In [82]:
import csv

def parseCSV(filename: str) -> list[dict]:
    with open(filename, "r", newline="") as file:
        reader = csv.DictReader(file)
        data = list(reader)
    return data

In [83]:
def traffic_flow_analyzer(trafficData: list[dict]) -> dict:
    # Analyze traffic flow data and return insights
    carsPerHour: dict[str, int] = {}
    for entry in trafficData:
        hour = entry["hour"]
        cars = int(entry["cars"])
        if hour not in carsPerHour:
            carsPerHour[hour] = cars
        else:
            carsPerHour[hour] += cars
    return carsPerHour

In [84]:
def calculate_peak_traffic(trafficData: list[dict], numPeakHours: int = 5) -> str:
    carsPerHour = traffic_flow_analyzer(trafficData)

    peakHours = []
    for i in range(numPeakHours):
        peakHour = max(carsPerHour, key=carsPerHour.get)
        peakHours.append(peakHour)
        del carsPerHour[peakHour]

    return peakHours
    # peakHour = max(carsPerHour, key=carsPerHour.get)
    # return peakHour

In [85]:
def mostCongestedStreet(trafficData: list[dict]) -> str:
    # Analyze traffic data to find the most congested street
    congestionPerStreet: dict[str, int] = {}
    for entry in trafficData:
        street = entry["road_id"]
        cars = int(entry["cars"])
        if street not in congestionPerStreet:
            congestionPerStreet[street] = cars
        else:
            congestionPerStreet[street] += cars

    # for congestion in congestionPerStreet:
    #     print(f"Street: {congestion}, Total Cars: {congestionPerStreet[congestion]}")

    mostCongested = max(congestionPerStreet, key=congestionPerStreet.get)
    return mostCongested

In [86]:
def avgHourlyTrafficPerDay(trafficData: list[dict]) -> dict:
    # Calculate average traffic per day
    trafficPerDay: dict[str, list[int]] = {}
    for entry in trafficData:
        day = entry["day"]
        cars = int(entry["cars"])
        if day not in trafficPerDay:
            trafficPerDay[day] = [cars]
        else:
            trafficPerDay[day].append(cars)

    avgTraffic: dict[str, float] = {}
    for day, carsList in trafficPerDay.items():
        avgTraffic[day] = sum(carsList) / len(carsList)

    return avgTraffic

In [87]:
def detectTrafficAnomalies(trafficData: list[dict]) -> dict[int, int]:
    # Build a day -> hour -> total cars mapping.
    totals_by_day: dict[str, dict[int, int]] = {}
    # Iterate over each CSV row.
    for entry in trafficData:
        # Read the day label.
        day = entry["day"]
        # Read the hour as an integer.
        hour = int(entry["hour"])
        # Read the cars count as an integer.
        cars = int(entry["cars"])
        # Ensure the day key exists.
        if day not in totals_by_day:
            # Initialize the day map.
            totals_by_day[day] = {}
        # Initialize the hour bucket if missing.
        if hour not in totals_by_day[day]:
            # First value for this hour.
            totals_by_day[day][hour] = cars
        else:
            # Accumulate cars for this hour.
            totals_by_day[day][hour] += cars

    # Initialize the 24-hour diff sums.
    diff_sums: dict[int, int] = {hour: 0 for hour in range(24)}
    # Walk each day independently.
    for day, hour_map in totals_by_day.items():
        # Consider all hours present for this day.
        for hour in sorted(hour_map):
            # Compute the left neighbor hour.
            left_hour = hour - 1
            # Compute the right neighbor hour.
            right_hour = hour + 1
            # Use only hours that have both neighbors.
            if left_hour in hour_map and right_hour in hour_map:
                # Read left neighbor traffic.
                left = hour_map[left_hour]
                # Read right neighbor traffic.
                right = hour_map[right_hour]
                # Compute neighbor difference.
                diff = right - left
                # Accumulate into the global sum for this hour.
                diff_sums[hour] += diff

    # Print the 24-hour summary.
    for hour in range(24):
        # Print one line per hour.
        print(f"Hour: {hour}, DiffSum: {diff_sums[hour]}")
    # Return the dictionary with 24 entries.
    return diff_sums

In [ ]:
def main():
    data = parseCSV("Datasets/traffic_flow.csv")
    # for entry in data:
    #     print(entry)
    
    analyzedData = traffic_flow_analyzer(data)
    for hour, cars in analyzedData.items():
        print(f"Hour: {hour}, Total Cars: {cars}")
    numPeakHours = 4
    print(f"Top {numPeakHours} Peak Hours:", calculate_peak_traffic(data, numPeakHours))
    print("Most Congested Street:", mostCongestedStreet(data))
    print("Average Traffic Per Day:", avgHourlyTrafficPerDay(data))
    # print("Traffic Anomalies:", detectTrafficAnomalies(data))

if __name__ == "__main__":
    main()

Hour: 0, Total Cars: 4201
Hour: 1, Total Cars: 4825
Hour: 2, Total Cars: 4226
Hour: 3, Total Cars: 4032
Hour: 4, Total Cars: 4247
Hour: 5, Total Cars: 4561
Hour: 6, Total Cars: 4045
Hour: 7, Total Cars: 4363
Hour: 8, Total Cars: 11781
Hour: 9, Total Cars: 11000
Hour: 10, Total Cars: 4329
Hour: 11, Total Cars: 4113
Hour: 12, Total Cars: 4322
Hour: 13, Total Cars: 4656
Hour: 14, Total Cars: 4626
Hour: 15, Total Cars: 4469
Hour: 16, Total Cars: 3967
Hour: 17, Total Cars: 11532
Hour: 18, Total Cars: 11363
Hour: 19, Total Cars: 4382
Hour: 20, Total Cars: 4288
Hour: 21, Total Cars: 4357
Hour: 22, Total Cars: 4457
Hour: 23, Total Cars: 4286
Top 4 Peak Hours: ['8', '17', '18', '9']
Most Congested Street: R4
Average Traffic Per Day: {'Mon': 160.54166666666666, 'Tue': 153.66666666666666, 'Wed': 169.08333333333334, 'Thu': 159.725, 'Fri': 157.15, 'Sat': 150.64166666666668, 'Sun': 152.75833333333333}
Hour: 0, DiffSum: 0
Hour: 1, DiffSum: 25
Hour: 2, DiffSum: -793
Hour: 3, DiffSum: 21
Hour: 4, DiffS